In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import roc_curve, auc
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm

In [8]:
#отобрали только предикторы у которых  p-value <0.05
predict_df = pd.read_excel('датафрейм с предикторами 19_04_26 с Killip.xlsx')
predict_df = predict_df.drop(columns='Unnamed: 0', errors='ignore')

print(predict_df['КШ развился в реанимации'].value_counts())
#выводим результат
print(predict_df.shape)
print(predict_df.dtypes)  #выводит типы данных каждого столбца

КШ развился в реанимации
0    5682
1     200
Name: count, dtype: int64
(5882, 198)
Количество пораженных сосудов(Syntax)_0          int64
Количество пораженных сосудов(Syntax)_1          int64
Количество пораженных сосудов(Syntax)_2          int64
Количество пораженных сосудов(Syntax)_3          int64
Количество пораженных сосудов(Syntax)_4          int64
                                                ...   
Медицинская помощь оказана за первые 4 часа    float64
PAMI (Средний риск)                            float64
Гипертоническая болезнь                          int64
TIMI категория                                 float64
КШ развился в реанимации                         int64
Length: 198, dtype: object


In [9]:
def compute_confidence_interval(data, confidence=0.95):
    mean = np.mean(data)
    sem = stats.sem(data)
    interval = sem * stats.t.ppf((1 + confidence) / 2, len(data) - 1)
    return mean, mean - interval, mean + interval

def format_confidence_interval(mean, lower, upper):
    return f"{mean:.4f} [{lower:.4f}; {upper:.4f}]"


features = [col for col in predict_df.columns if col != 'КШ развился в реанимации']  #исключаем целевую переменную сразу


predict_df = predict_df.dropna()  #удаляем строки с NaN значениями

#подготовка данных
X = predict_df[features]
y = predict_df['КШ развился в реанимации']


results_data = []  #список для хранения результатов по каждому признаку
n_repeats = 100  #количество повторений разбиения на 80/20
for feature in features:
    print(f"Оценка для признака: {feature}")
    x_feature = X[[feature]] #используем DataFrame для одного признака

    all_roc_auc_test = [] #хранит все roc_auc на тестовой выборке
    all_coef = [] #хранит все веса



    # 1. Считаем p-value на всей выборке через statsmodels
    # Добавляем константу (интерцепт), так как statsmodels не добавляет её сам
    X_with_const = sm.add_constant(x_feature)
    try:
        logit_model = sm.Logit(y, X_with_const).fit(disp=0)
        p_value = logit_model.pvalues[feature]
    except:
        p_value = np.nan # На случай ошибок сходимости




    for i in range(n_repeats):
        np.random.seed(i + 42)
        #разделение на обучающую и тестовую выборки (80/20)
        x_train, x_test, y_train, y_test = train_test_split(x_feature, y, train_size=0.8, stratify=y, random_state=i + 42)

        model = LogisticRegression(solver='lbfgs', max_iter=2000, C=1, penalty='l2')

        #обучение модели на всей обучающей выборке (x_train)
        model.fit(x_train, y_train)
        all_coef.append(model.coef_[0][0])
        #тестирование на отложенной выборке (x_test)
        y_pred_prob = model.predict_proba(x_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
        roc_auc = auc(fpr, tpr)
        all_roc_auc_test.append(roc_auc)

    #расчет доверительных интервалов
    roc_auc_test_mean, roc_auc_test_lower, roc_auc_test_upper = compute_confidence_interval(all_roc_auc_test)
    coef_mean, coef_lower, coef_upper = compute_confidence_interval(all_coef)


    results_data.append({
        'Признак': feature,
        'Весовой коэф-т [95% ДИ]': format_confidence_interval(coef_mean, coef_lower, coef_upper),
        'p-value': f"{p_value:.4f}",
        'Тестовый ROC-AUC [95% ДИ]': format_confidence_interval(roc_auc_test_mean, roc_auc_test_lower, roc_auc_test_upper)
    })

#создание DataFrame после цикла
results_df = pd.DataFrame(results_data)
# Удобное форматирование p-value (замена 0.0000 на <0.001)
results_df['p-value'] = results_df['p-value'].apply(lambda x: "<0.00001" if x == "0.000000" else x)
print(results_df)
results_df.to_excel('однофакторные модели от 14_03_26.xlsx', index=False)

Оценка для признака: Количество пораженных сосудов(Syntax)_0
Оценка для признака: Количество пораженных сосудов(Syntax)_1
Оценка для признака: Количество пораженных сосудов(Syntax)_2
Оценка для признака: Количество пораженных сосудов(Syntax)_3
Оценка для признака: Количество пораженных сосудов(Syntax)_4
Оценка для признака: Количество пораженных сосудов(Значимость)_0
Оценка для признака: Количество пораженных сосудов(Значимость)_1
Оценка для признака: Количество пораженных сосудов(Значимость)_2
Оценка для признака: Количество пораженных сосудов(Значимость)_3
Оценка для признака: CADILLAC категория_Высокий уровень
Оценка для признака: CADILLAC категория_Неизвестно
Оценка для признака: CADILLAC категория_Низкий уровень
Оценка для признака: CADILLAC категория_Средний уровень
Оценка для признака: РЕКОРД категория_Высокий риск
Оценка для признака: РЕКОРД категория_Неизвестно
Оценка для признака: РЕКОРД категория_Низкий риск
Оценка для признака: РЕКОРД категория_Очень высокий
Оценка для приз

/Users/MAC/PycharmProjects/диплом от 15.11/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Оценка для признака: TIMI Летальность категория_Умеренный риск
Оценка для признака: Вид операции(ИБ)(Новый)_-
Оценка для признака: Вид операции(ИБ)(Новый)_nan
Оценка для признака: Вид операции(ИБ)(Новый)_Аспирация тромботических масс


/Users/MAC/PycharmProjects/диплом от 15.11/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Оценка для признака: Вид операции(ИБ)(Новый)_Имплантация
Оценка для признака: Вид операции(ИБ)(Новый)_Лапароскопическая аппендектомия
Оценка для признака: Вид операции(ИБ)(Новый)_Реваскуляризация
Оценка для признака: Вид операции(ИБ)(Новый)_Реканализация


...

Оценка для признака: TIMI (Летальность) (Низкий риск)
Оценка для признака: Стенокардия в диагнозе при поступлении


/Users/MAC/PycharmProjects/диплом от 15.11/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Оценка для признака: ХОБЛ
Оценка для признака: Тромболизис
Оценка для признака: Общий анализ крови_экспресс раньше операции
Оценка для признака: СД
Оценка для признака: ФП a (в анамнезе)
Оценка для признака: ХБП


/Users/MAC/PycharmProjects/диплом от 15.11/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Оценка для признака: PAMI (Низкий риск)
Оценка для признака: ПИКС в диагнозе при поступлении
Оценка для признака: Левосимендан
Оценка для признака: Медицинская помощь оказана за первые 4 часа
Оценка для признака: PAMI (Средний риск)
Оценка для признака: Гипертоническая болезнь
Оценка для признака: TIMI категория
                                         Признак     Весовой коэф-т [95% ДИ]  \
0        Количество пораженных сосудов(Syntax)_0     0.0000 [0.0000; 0.0000]   
1        Количество пораженных сосудов(Syntax)_1     0.3791 [0.3373; 0.4208]   
2        Количество пораженных сосудов(Syntax)_2  -0.7226 [-0.7638; -0.6813]   
3        Количество пораженных сосудов(Syntax)_3     0.3124 [0.2683; 0.3565]   
4        Количество пораженных сосудов(Syntax)_4     0.0000 [0.0000; 0.0000]   
..                                           ...                         ...   
192                                 Левосимендан     0.0000 [0.0000; 0.0000]   
193  Медицинская помощь оказана за первые 4 ча